<a href="https://colab.research.google.com/github/EvolvingAgentsLabs/agent-forge/blob/main/jit_poc_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================================================
# CELL 1: Environment Setup
# ===================================================
# @title Step 1: Install Unsloth & Dependencies

# We will use Unsloth's optimized installation for Colab.
# This provides significant speed and memory improvements.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install other necessary libraries
!pip install -q --no-deps transformers peft accelerate bitsandbytes
!pip install -q requests

print("✅ Dependencies installed with Unsloth.")

# Check the allocated GPU. A T4 or L4 from a free Colab instance is sufficient.
!nvidia-smi

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-24g8555z/unsloth_d51a323264b44878971c5e67725713fb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-24g8555z/unsloth_d51a323264b44878971c5e67725713fb
  Resolved https://github.com/unslothai/unsloth.git to commit dc26a7a0eb20c31549318396f53639ba8c01025e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Dependencies installed with Unsloth.
Sun Aug 17 18:11:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/C

In [ ]:
# ===================================================
# CELL 2: Load the Base Model
# ===================================================
# @title Step 2: Load Qwen2.5-Coder-1.5B with 4-bit Quantization

from unsloth import FastQwen2Model
import torch

# The maximum sequence length the model can handle.
# Qwen2.5-Coder supports long context, which is great for complex tasks.
max_seq_length = 8192

# Load the model and tokenizer using Unsloth's highly optimized function.
# This applies 4-bit quantization to fit the model into Colab's GPU memory.
model, tokenizer = FastQwen2Model.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,      # None for auto detection of the best data type
    load_in_4bit = True,
)

print("✅ Qwen2.5-Coder-1.5B model loaded successfully with Unsloth.")

==((====))==  Unsloth 2025.8.6: Fast Qwen2 patching. Transformers: 4.55.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Qwen2.5-Coder-1.5B model loaded successfully with Unsloth.


In [ ]:
# ===================================================
# CELL 3: Prepare the Model for Fine-Tuning
# ===================================================
# @title Step 3: Prepare the Model for Fine-Tuning with LoRA

# This step injects the LoRA adapters into the model, making it trainable
# in a parameter-efficient way.
model = FastQwen2Model.get_peft_model(
    model,
    r = 16, # Rank of the LoRA update matrices. 16 is a good starting point.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # Target all linear layers for comprehensive fine-tuning.
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # A key Unsloth feature for memory saving
    random_state = 3407,
)

print("✅ Model prepared for LoRA fine-tuning.")

✅ Model prepared for LoRA fine-tuning.


In [ ]:
# ===================================================
# CELL 4: The Fine-Tuning Dataset
# ===================================================
# @title Step 4: Define an Expanded and More Robust Fine-Tuning Dataset

import json
from datasets import load_dataset, Dataset

# This expanded dataset is crucial. It provides varied examples to teach
# the LORA its specific job as a "Concept-to-Code" translator, preventing it
# from hallucinating or failing to follow instructions.
raw_dataset = [
    {
        "messages": [
            {"role": "user", "content": "<FUNC_START>\nfunction get_weather(city: str):\n    \"\"\"Fetches current weather for a city using Open-Meteo API. Returns JSON with 'temperature' and 'condition'.\"\"\"\n    # Get lat/lon, then call weather API.\n<FUNC_END>"},
            {"role": "assistant", "content": "import requests\n\ndef get_weather(city: str) -> dict:\n    \"\"\"Fetches current weather for a city using Open-Meteo API.\"\"\"\n    try:\n        geo_url = f\"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1\"\n        geo_res = requests.get(geo_url)\n        geo_res.raise_for_status()\n        geo_data = geo_res.json()['results'][0]\n        lat, lon = geo_data['latitude'], geo_data['longitude']\n        weather_url = f\"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,weather_code\"\n        weather_res = requests.get(weather_url)\n        weather_res.raise_for_status()\n        current = weather_res.json()['current']\n        return {\"temperature\": current['temperature_2m'], \"condition_code\": current['weather_code']}\n    except Exception as e:\n        return {\"error\": str(e)}\n"}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "<FUNC_START>\nfunction get_random_joke():\n    \"\"\"Fetches a random joke from the official-joke-api.\"\"\"\n    # Call the API and return the setup and punchline.\n<FUNC_END>"},
            {"role": "assistant", "content": "import requests\n\ndef get_random_joke() -> str:\n    \"\"\"Fetches a random joke from the official-joke-api.\"\"\"\n    try:\n        response = requests.get('https://official-joke-api.appspot.com/random_joke')\n        response.raise_for_status()\n        joke_data = response.json()\n        return f\"{joke_data['setup']} - {joke_data['punchline']}\"\n    except Exception as e:\n        return f\"Could not fetch joke: {e}\"\n"}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "<FUNC_START>\nfunction save_report(filename: str, report_content: str):\n    \"\"\"Saves a report to a local file.\"\"\"\n    # 1. Open the file.\n    # 2. Write the content.\n<FUNC_END>"},
            {"role": "assistant", "content": "import os\n\ndef save_report(filename: str, report_content: str) -> dict:\n    \"\"\"Saves a report to a local file.\"\"\"\n    try:\n        with open(filename, 'w', encoding='utf-8') as f:\n            f.write(report_content)\n        return {\"status\": \"success\", \"path\": os.path.abspath(filename)}\n    except Exception as e:\n        return {\"status\": \"error\", \"message\": str(e)}\n"}
        ]
    }
]

dataset = Dataset.from_list(raw_dataset)

# This function formats our raw data into the specific chat template
# that the Qwen model was trained on, which is critical for good performance.
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = True) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

print(f"✅ Expanded dataset with {len(raw_dataset)} examples created and formatted.")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ Expanded dataset with 3 examples created and formatted.


In [ ]:
# ===================================================
# CELL 5: Fine-Tune the Model
# ===================================================
# @title Step 5: Train the LoRA Adapter

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100, # Increased steps for better learning on the expanded dataset
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting LoRA fine-tuning...")
trainer.train()

LORA_ADAPTER_PATH = "adapters/agent_forge_translator"
trainer.save_model(LORA_ADAPTER_PATH)

print(f"\n✅ LoRA adapter fine-tuned and saved to {LORA_ADAPTER_PATH}")

Unsloth: Tokenizing ["text"]:   0%|          | 0/3 [00:00<?, ? examples/s]

Starting LoRA fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 100 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,2.083900
2,2.083900
3,1.993600
4,1.771100
5,1.537000
6,1.285700
7,1.018000
8,0.816300
9,0.669500
10,0.580900



✅ LoRA adapter fine-tuned and saved to adapters/agent_forge_translator


In [ ]:
# ===================================================
# CELL 6: The Autonomous, Self-Correcting Runtime
# ===================================================
# @title Step 6: Define the Autonomous Agent Forge Runtime (Final Version)

import re
import time
import json
from unsloth import FastQwen2Model

class AgentForgeRuntime:
    def __init__(self, finetuned_model, tokenizer):
        self.model = finetuned_model
        self.tokenizer = tokenizer
        FastQwen2Model.for_inference(self.model)
        self.function_cache = {}
        print("✅ Autonomous Agent Forge Runtime Initialized.")

    def _call_model(self, prompt, **generation_kwargs):
        print(f"\n>>> Calling Fine-Tuned Model...")
        messages = [{"role": "user", "content": prompt}]
        inputs = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
        ).to("cuda")
        default_kwargs = {
            "max_new_tokens": 1024, "use_cache": True, "do_sample": True, "temperature": 0.1,
        }
        default_kwargs.update(generation_kwargs)
        if not default_kwargs.get("do_sample"):
            default_kwargs.pop("temperature", None)
        start_time = time.time()
        outputs = self.model.generate(**inputs, **default_kwargs)
        content = self.tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        latency = (time.time() - start_time) * 1000
        total_tokens = len(inputs["input_ids"][0]) + len(outputs[0][inputs["input_ids"].shape[-1]:])
        print(f"<<< Model responded in {latency:.2f}ms. (Total tokens: {total_tokens})")
        return content, latency, total_tokens

    def _compile_and_cache(self, concept, function_name, function_signature):
        print(f"--- Translating concept for '{function_name}' ---")
        translation_prompt = f"""You are an expert Python code generator. Your sole task is to translate the following function concept into a complete, executable Python function.
It is CRITICAL that the function you generate has the exact signature: `{function_signature}`.
Do not add any commentary. Only output the Python code.

Instruction:
{concept}
Output:"""

        full_output, _, _ = self._call_model(translation_prompt, do_sample=False)
        python_code = full_output
        print(f"--- Generated Python Code ---\n{python_code}\n-----------------------------")

        namespace = {"requests": __import__("requests"), "call_llm": self._call_model}
        try:
            exec(python_code, namespace)
            if function_name in namespace:
                self.function_cache[function_name] = namespace[function_name]
                print(f"--- Tool '{function_name}' compiled and cached. ---")
            else:
                actual_name_match = re.search(r"def\s+(\w+)\s*\(", python_code)
                if actual_name_match:
                    actual_name = actual_name_match.group(1)
                    print(f"--- Found function '{actual_name}' instead. Caching it as '{function_name}'. ---")
                    self.function_cache[function_name] = namespace[actual_name]
                else:
                    raise KeyError("The generated code did not define any valid function.")
        except Exception as e:
            raise ValueError(f"Translator LORA failed to generate a usable function for '{function_name}'. Error: {e}")

    # --- THE FINAL, SELF-CORRECTING EXECUTION LOOP ---
    def run_goal(self, goal, max_retries=2):
        total_latency, total_tokens = 0, 0

        print("\n" + "="*20 + " PHASE 1: PLANNING " + "="*20)
        plan_prompt = f"""You are a world-class AI planner. Your job is to create a plan to solve the user's goal.
The final step in your plan must ALWAYS be an action called 'final_answer'.
Output a JSON list of steps. Each step must have an 'id', 'action' ('execute_tool' or 'final_answer'), a 'tool_name' (if applicable), 'description' for the tool, and 'inputs' which is a dictionary.
User Goal: '{goal}'"""

        plan_str, lat, tok = self._call_model(plan_prompt, do_sample=False)
        total_latency += lat; total_tokens += tok

        try:
            clean_plan_str = re.sub(r"```json\n|```", "", plan_str).strip()
            plan = json.loads(clean_plan_str)
            print(f"--- Plan Generated ---\n{json.dumps(plan, indent=2)}\n----------------------")
        except json.JSONDecodeError:
            print(f"!!! CRITICAL ERROR: Orchestrator failed to generate a valid JSON plan. Raw output:\n{plan_str}")
            return "Failed to create a plan.", total_latency, total_tokens

        print("\n" + "="*20 + " PHASE 2: EXECUTION LOOP " + "="*20)
        state = {"goal": goal}

        for step in plan:
            step_id = step.get("id")
            action = step.get("action")

            inputs = {k: state.get(v.split('.')[1]) if isinstance(v, str) and v.startswith("state.") else v for k, v in step.get("inputs", {}).items()}

            if action == "execute_tool":
                tool_name = step["tool_name"]
                error_context = None # Reset for each new tool

                for attempt in range(max_retries):
                    try:
                        # THE CORE FIX: The concept generation is now inside the retry loop
                        # so it can be improved with error context on subsequent attempts.
                        if tool_name not in self.function_cache:
                            input_params_str = ", ".join([f"{k}: {type(v).__name__}" for k, v in inputs.items()])
                            function_signature = f"def {tool_name}({input_params_str}):"
                            docstring = step.get('description', 'A tool to help achieve the goal.')

                            # Construct the base concept
                            concept = f"<FUNC_START>\nfunction {tool_name}({input_params_str}):\n    \"\"\"{docstring}\"\"\"\n    # Implement the logic for this function.\n<FUNC_END>"

                            # If this is a retry, inject the error context as a hint
                            if error_context:
                                concept += f"\n\n# SELF-CORRECTION HINT: The last attempt to run this tool failed with the error: '{error_context}'. Please analyze this error and generate a new version of the code that fixes it. For example, if a variable was not defined, ensure you define it before it is used (e.g., by making a necessary API call)."

                            self._compile_and_cache(concept, tool_name, function_signature)

                        tool_func = self.function_cache[tool_name]
                        start_exec = time.time()
                        result = tool_func(**inputs)
                        exec_latency = (time.time() - start_exec) * 1000
                        total_latency += exec_latency

                        if isinstance(result, dict) and "error" in result:
                            raise Exception(result["error"])

                        state[f"result_of_{step_id}"] = result
                        print(f"--- Step '{step_id}' Result ---\n{result}\n--------------------------")
                        break

                    except Exception as e:
                        error_context = str(e) # Save the error for the next attempt's prompt
                        print(f"!!! ATTEMPT {attempt + 1}/{max_retries}: Tool '{tool_name}' failed. Error: {error_context}")
                        state[f"error_of_{step_id}_attempt_{attempt+1}"] = error_context
                        self.function_cache.pop(tool_name, None) # Clear the bad tool from cache
                        if attempt + 1 >= max_retries:
                            print(f"!!! CRITICAL FAILURE: Tool '{tool_name}' failed after {max_retries} attempts. Aborting.")
                            return f"Agent failed on step {step_id}.", total_latency, total_tokens

            elif action == "final_answer":
                print("\n" + "="*20 + " PHASE 3: FINAL SYNTHESIS " + "="*20)
                final_prompt = f"""Based on the user's goal '{state['goal']}' and the data gathered: {json.dumps(state)}, write a final, natural language response to the user. Do not output code or a plan. Just provide the answer.
Answer:"""

                final_answer, lat, tok = self._call_model(final_prompt)
                total_latency, total_tokens = total_latency + lat, total_tokens + tok

                print("\n" + "="*20 + " FINAL ANSWER " + "="*20)
                print(final_answer)
                return final_answer, total_latency, total_tokens

        return "The plan completed without providing a final answer.", total_latency, total_tokens

In [ ]:
# ===================================================
# CELL 7: The Benchmark Harness
# ===================================================
# @title Step 7: Define and Run the Benchmark

def main():
    print("="*60 + "\n        RUNNING AUTONOMOUS AGENT (CLOSED LOOP) BENCHMARK\n" + "="*60)

    agent_forge_runtime = AgentForgeRuntime(
        finetuned_model = model,
        tokenizer = tokenizer
    )

    user_goal = "First, find out the current temperature in Berlin. Second, get a random joke. Finally, tell me the temperature and the joke in a single sentence."

    final_answer, total_latency, total_tokens = agent_forge_runtime.run_goal(user_goal)

    print("\n" + "="*60 + "\n                  BENCHMARK SUMMARY\n" + "="*60)
    print(f"Final Answer: {final_answer}")
    print(f"Total Latency: {total_latency:,.2f} ms")
    print(f"Total Tokens (est.): {total_tokens}")
    print("="*60)

# --- Run the main function ---
main()

        RUNNING AUTONOMOUS AGENT (CLOSED LOOP) BENCHMARK
✅ Autonomous Agent Forge Runtime Initialized.

==================== PHASE 1: PLANNING ====================

>>> Calling Fine-Tuned Model...
<<< Model responded in 8543.08ms. (Total tokens: 281)
--- Plan Generated ---
[
  {
    "id": "1",
    "action": "execute_tool",
    "tool_name": "geo_weather",
    "description": "Fetches current weather for a city using Open-Meteo API.",
    "inputs": {
      "city": "Berlin"
    }
  },
  {
    "id": "2",
    "action": "execute_tool",
    "tool_name": "joke_fetcher",
    "description": "Fetches a random joke using official-joke-api.",
    "inputs": {}
  },
  {
    "id": "3",
    "action": "final_answer",
    "result_path": "weather_joke.txt",
    "message": "Found weather and joke."
  }
]
----------------------

==================== PHASE 2: EXECUTION LOOP ====================
--- Translating concept for 'geo_weather' ---

>>> Calling Fine-Tuned Model...
<<< Model responded in 8272.10ms. (To